# 15: Turn a graph problem into a quantum objective

**Level:** Intermediate to advanced  
**Before you start:** Notebook 03; basic graph intuition.  
**Resources:** CPU unless an optional remote step is enabled.

Split four vertices into two groups so that as many edges as possible cross between the groups.

Run each cell in order. All core calculations are written in this notebook.

## 1. Enumerate a tiny classical reference

The graph is a square. A bit string assigns each vertex to group 0 or 1. An edge contributes one when its endpoints differ.

In [ ]:
import itertools
import torch
import flagquantum as fq
from flagquantum.algorithms import Hamiltonian, pauli_term
import matplotlib.pyplot as plt

edges = [(0, 1), (1, 2), (2, 3), (3, 0)]
strings = list(itertools.product([0, 1], repeat=4))
scores = torch.tensor(
    [sum(bits[i] != bits[j] for i, j in edges) for bits in strings], dtype=torch.float32
)
print("Best cut:", scores.max().item())
print(
    "Optimal assignments:",
    [bits for bits, score in zip(strings, scores) if score == scores.max()],
)


## 2. Build one QAOA layer

An edge cut is (1-ZᵢZⱼ)/2. The cost unitary uses CNOT–RZ–CNOT, followed by RX mixing gates. We omit only the global phase from the identity term. This is a small educational QAOA circuit, not a speedup claim.

In [ ]:
def qaoa(parameters):
    gamma, beta = parameters
    q = fq.Circuit(4)
    for wire in range(4):
        q.h(wire)
    for i, j in edges:
        q.cx(i, j)
        q.rz(j, -gamma)
        q.cx(i, j)
    for wire in range(4):
        q.rx(wire, 2 * beta)
    return q


def expected_cut(parameters):
    state = fq.run(qaoa(parameters)).to_statevector().reshape(-1)
    return (state.abs().square() * scores).sum()


parameters = torch.nn.Parameter(torch.tensor([0.4, 0.2]))
optimizer = torch.optim.Adam([parameters], lr=0.05)


## 3. Maximize expected cut size

Minimizing the negative score maximizes the score. A high expected score does not guarantee every sampled assignment is optimal.

In [ ]:
history = []
for step in range(100):
    optimizer.zero_grad()
    score = expected_cut(parameters)
    (-score).backward()
    optimizer.step()
    history.append(score.item())
print("Final expected cut:", expected_cut(parameters).item())
assert expected_cut(parameters).item() > 2.5
probabilities = (
    fq.run(qaoa(parameters.detach()))
    .to_statevector()
    .reshape(-1)
    .abs()
    .square()
    .detach()
)
fig, ax = plt.subplots(1, 2, figsize=(11, 3))
ax[0].plot(history)
ax[0].axhline(scores.max().item(), linestyle="--", color="black")
ax[0].set(xlabel="Step", ylabel="Expected cut")
ax[1].bar(["".join(map(str, b)) for b in strings], probabilities.tolist())
ax[1].tick_params(axis="x", rotation=90)
ax[1].set_ylabel("Probability")
plt.tight_layout()
plt.show()


## Make it yours

Change the graph to a triangle or a path, updating qubit count and reference enumeration together. Add another QAOA layer and compare several seeds. Report both expected score and probability of sampling an optimal assignment.